**Installing all libraries**

In [2]:
%%capture
%pip install -U \
langchain \
langchain-core \
langchain-community \
langchain-text-splitters \
langchain-huggingface \
langchain-chroma \
langchain-classic \
langchain-ibm \
ibm-watsonx-ai \
chromadb \
sentence-transformers \
transformers \
huggingface-hub \
wget

In [1]:
%pip install --upgrade \
numpy \
pandas \
scipy \
scikit-learn

**Importing all Libraries**

In [2]:
def warn(*args, **kwargs):
    pass

import warnings
warnings.warn = warn
warnings.filterwarnings("ignore")

import wget

# LangChain
from langchain_community.document_loaders import TextLoader
from langchain_text_splitters import CharacterTextSplitter
from langchain_chroma import Chroma
from langchain_huggingface import HuggingFaceEmbeddings
from langchain_classic.chains import RetrievalQA, ConversationalRetrievalChain
from langchain_classic.memory import ConversationBufferMemory
from langchain_core.prompts import PromptTemplate

# IBM watsonx
from ibm_watsonx_ai.foundation_models import Model
from ibm_watsonx_ai.metanames import GenTextParamsMetaNames as GenParams
from ibm_watsonx_ai.foundation_models.utils.enums import (
    ModelTypes,
    DecodingMethods,
)
from langchain_ibm import WatsonxLLM

print("All imports successful!")

All imports successful!


**Preprocessing**


**Loading the Document**
 : This is the load step in Indexing.

In [3]:
filename = 'companyPolicies.txt'
url = 'https://cf-courses-data.s3.us.cloud-object-storage.appdomain.cloud/6JDbUb_L3egv_eOkouY71A.txt'

# Use wget to download the file
wget.download(url, out=filename)
print('file downloaded')

file downloaded


In [4]:
with open(filename, 'r') as file:
  contents = file.read()
  print(contents)

1.	Code of Conduct

Our Code of Conduct outlines the fundamental principles and ethical standards that guide every member of our organization. We are committed to maintaining a workplace that is built on integrity, respect, and accountability.
Integrity: We hold ourselves to the highest ethical standards. This means acting honestly and transparently in all our interactions, whether with colleagues, clients, or the broader community. We respect and protect sensitive information, and we avoid conflicts of interest.
Respect: We embrace diversity and value each individual's contributions. Discrimination, harassment, or any form of disrespectful behavior is unacceptable. We create an inclusive environment where differences are celebrated and everyone is treated with dignity and courtesy.
Accountability: We take responsibility for our actions and decisions. We follow all relevant laws and regulations, and we strive to continuously improve our practices. We report any potential violations of 

**Splitting the Documents into Chunks**


LangChain is used to split the document and create chunks. It helps you divide a long story (document) into smaller parts, which are called chunks, so that it's easier to handle.

For the splitting process, the goal is to ensure that each segment is as extensive as if you were to count to a certain number of characters and meet the split separator. This certain number is called chunk size. Let's set 1000 as the chunk size in this project. Though the chunk size is 1000, the splitting is happening randomly. This is an issue with LangChain. CharacterTextSplitter uses \n\n as the default split separator. You can change it by adding the separator parameter in the CharacterTextSplitter function; for example, separator="\n".

In [5]:
loader = TextLoader(filename)
documents = loader.load()
text_splitter = CharacterTextSplitter(chunk_size=1000, chunk_overlap=0)
texts = text_splitter.split_documents(documents)
print(len(texts))

16


**Embedding and Storing**


This step is the embed and store processes in Indexing.In this step, you're taking the pieces of the story, your "chunks," converting the text into numbers, and making them easier for your computer to understand and remember by using a process called "embedding." Think of embedding like giving each chunk its own special code. This code helps the computer quickly find and recognize each chunk later on.

In [6]:
embeddings = HuggingFaceEmbeddings()
docsearch = Chroma.from_documents(texts, embeddings)
print('document ingested')

modules.json:   0%|          | 0.00/349 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/116 [00:00<?, ?B/s]

README.md:   0%|          | 0.00/11.6k [00:00<?, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/53.0 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/571 [00:00<?, ?B/s]

...

vocab.txt:   0%|          | 0.00/232k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/466k [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/239 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/190 [00:00<?, ?B/s]

document ingested


**LLM MODEL CONSTRUCTION**


In [7]:
model_id = 'meta-llama/llama-4-maverick-17b-128e-instruct-fp8'

In [8]:
parameters = {
    GenParams.DECODING_METHOD: DecodingMethods.GREEDY,
    GenParams.MAX_NEW_TOKENS: 256,
    GenParams.MIN_NEW_TOKENS: 130,
    GenParams.TEMPERATURE: 0.5
}

In [14]:
credentials = {
    "url": "https://us-south.ml.cloud.ibm.com"
}

project_id = "skills-network"

In [15]:
llm = WatsonxLLM(
    model_id=model_id,
    url="https://us-south.ml.cloud.ibm.com",
    project_id=project_id,
    params=parameters,
)

ValidationError: 1 validation error for WatsonxLLM
  Value error, Did not find 'api_key' or 'token', please add an environment variable `WATSONX_API_KEY` or 'WATSONX_TOKEN' which contains it, or pass 'api_key' or 'token' as a named parameter. [type=value_error, input_value={'model_id': 'meta-llama/...30, 'temperature': 0.5}}, input_type=dict]
    For further information visit https://errors.pydantic.dev/2.13/v/value_error

**Integrating LangChain**


LangChain has a number of components that are designed to help retrieve information from the document and build question-answering applications, which helps you complete the retrieve part of the Retrieval task.
split

In [12]:
qa = RetrievalQA.from_chain_type(llm=llm,
                                 chain_type="stuff",
                                 retriever=docsearch.as_retriever(),
                                 return_source_documents=False)
query = "what is mobile policy?"
qa.invoke(query)

NameError: name 'llm' is not defined